# Imports

In [47]:
import os
import time
import requests
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.decomposition import PCA
import umap
from sklearn.random_projection import GaussianRandomProjection
from sklearn.preprocessing import normalize

# Configuration

In [ ]:

NTFY_TOPIC = "aysel_tfe_server_9988" 
# "Flickr8k"  or "Flickr30k" or "ConceptualCaptions"
CURRENT_DATASET = "Flickr8k"  
DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]  

def send_ntfy(message, title="TFE Production"):
    try:
        requests.post(f"https://ntfy.sh/{NTFY_TOPIC}", 
                      data=message.encode('utf-8'),
                      headers={"Title": title, "Priority": "3"})
    except: pass

# Data Loading

In [49]:
def load_raw_matrix(modality, model_name):
    """Load raw embeddings from the dataset folder structure."""
    file_path = os.path.join(
        DATA_DIR,
        "Results",
        CURRENT_DATASET,
        "Features_Raw",
        modality,
        model_name,
        "embeddings.npy"
    )
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing RAW embeddings: {file_path}")
    return np.load(file_path)

In [50]:
def load_reduced(dataset, modality, model, method, dim):
    path = os.path.join(
        DATA_DIR,
        "Features_Reduced",
        f"X_{modality}_{model}_{method}_{dim}_{dataset}.npy"
    )

    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing reduced: {path}")

    return np.load(path)

# Dimension Reduction

In [51]:
def run_production_batch(modality, model_name):
    print(f"\nProcessing: {model_name.upper()} ({modality})")
    
    try:
        X_raw = load_raw_matrix(modality, model_name)
        X_raw = normalize(X_raw)  # améliore stabilité PCA/UMAP
    except Exception as e:
        print(f"[SKIP] {model_name}: {e}")
        return

    input_dim = X_raw.shape[1]

    for method_name, reduction_func in REDUCTION_PIPELINE.items():
        for dim in DIMENSIONS_TO_TEST:
            if dim >= input_dim:
                continue

            X_reduced = reduction_func(X_raw, dim)

            save_dir = os.path.join(REDUCED_DIR, CURRENT_DATASET, modality)
            os.makedirs(save_dir, exist_ok=True)

            save_filename = f"X_{modality}_{model_name}_{method_name}_{dim}_{CURRENT_DATASET}.npy"
            save_path = os.path.join(save_dir, save_filename)

            if os.path.exists(save_path):
                print(f"[SKIP] {save_filename}")
                continue

            np.save(save_path, X_reduced)

    print(f"Finished all reductions for {model_name}")
    send_ntfy(f"Reductions for {model_name} ({CURRENT_DATASET}) are saved to disk.")

# Execution

## Parameters

In [52]:
DIMENSIONS_TO_TEST = [512, 256, 128, 64, 32, 16]

REDUCTION_PIPELINE = {
    "pca": lambda X, d: PCA(n_components=d).fit_transform(X),
    "grp": lambda X, d: GaussianRandomProjection(n_components=d).fit_transform(X),
    "umap": lambda X, d: umap.UMAP(
        n_components=d,
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine"
    ).fit_transform(X)
}

VISION_MODELS = ["resnet50", "mobilenet_v3", "vit", "deit", "clip_vision"]
TEXT_MODELS = ["rnn", "bert", "roberta", "gpt2", "clip_text"]

## Run Experiment

In [53]:
def run_full_pipeline():

    start_global = time.time()

    print(f"\n===== STARTING REDUCTIONS: {CURRENT_DATASET} =====\n")

    # --- Vision ---
    for model in VISION_MODELS:
        try:
            run_production_batch("vision", model)
        except Exception as e:
            print(f"[ERROR] Vision {model}: {e}")

    # --- Text ---
    for model in TEXT_MODELS:
        try:
            run_production_batch("text", model)
        except Exception as e:
            print(f"[ERROR] Text {model}: {e}")

    total_time = time.time() - start_global

    print(f"\n===== DONE: {CURRENT_DATASET} in {total_time/60:.2f} min =====")

    send_ntfy(f"{CURRENT_DATASET} reductions COMPLETE in {total_time/60:.2f} min")

In [54]:
for CURRENT_DATASET in DATASETS:
    print(f"\n=== Starting Batch for Dataset: {CURRENT_DATASET} ===")
    run_full_pipeline()


=== Starting Batch for Dataset: Flickr8k ===

===== STARTING REDUCTIONS: Flickr8k =====


Processing: RESNET50 (vision)
Finished all reductions for resnet50

Processing: MOBILENET_V3 (vision)
Finished all reductions for mobilenet_v3

Processing: VIT (vision)
Finished all reductions for vit

Processing: DEIT (vision)
Finished all reductions for deit

Processing: CLIP_VISION (vision)
Finished all reductions for clip_vision

Processing: RNN (text)
Finished all reductions for rnn

Processing: BERT (text)
Finished all reductions for bert

Processing: ROBERTA (text)
Finished all reductions for roberta

Processing: GPT2 (text)
Finished all reductions for gpt2

Processing: CLIP_TEXT (text)
Finished all reductions for clip_text

===== DONE: Flickr8k in 20.61 min =====

=== Starting Batch for Dataset: Flickr30k ===

===== STARTING REDUCTIONS: Flickr30k =====


Processing: RESNET50 (vision)
Finished all reductions for resnet50

Processing: MOBILENET_V3 (vision)
Finished all reductions for mobile